In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [ ]:
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

start_date = "2013-01-01"
end_date   = "2023-01-01"

tikeri = {
    "WTI":   "CL=F",
    "BRENT": "BZ=F",
    "NVDA":  "NVDA"
}

podaci = {}
for naziv, tiker in tikeri.items():
    df = yf.download(tiker, start=start_date, end=end_date, interval="1wk",
                     progress=False, auto_adjust=False)
    podaci[naziv] = df['Close'].copy()

cene = pd.concat(podaci, axis=1)
cene.columns = tikeri.keys()
cene = cene.dropna()

print("Prvih 5 redova sirovih cena:")
print(cene.head())
print("\nDimenzije:", cene.shape)
cene = cene[cene > 0].dropna()

log_cene = np.log(cene)
log_cene.columns = [f"log_{c}" for c in log_cene.columns]

log_razlike = log_cene.diff().dropna()
log_razlike.columns = [f"Δ{col}" for col in log_razlike.columns]

print("\nPrvih 5 redova log-cena:")
print(log_cene.head(3))
print("\nPrvih 5 redova log-razlika (prinosa):")
print(log_razlike.head(3))

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

boje = {"log_WTI": "tab:blue", "log_BRENT": "tab:green", "log_NVDA": "tab:red"}

for ax, col in zip(axes, log_cene.columns):
    ax.plot(log_cene.index, log_cene[col], color=boje[col], linewidth=1.2)
    ax.set_title(f"Logaritamska cena – {col.replace('log_','')}", fontsize=13)
    ax.set_ylabel("ln(P_t)")

axes[-1].set_xlabel("Datum")
fig.suptitle("Vremenske serije logaritamskih cena (2013–2023)", fontsize=15, fontweight='bold')
plt.tight_layout()
# plt.savefig("log_cene.png", dpi=150)
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, col in zip(axes, log_razlike.columns):
    ax.plot(log_razlike.index, log_razlike[col],
            color=boje[col.replace('Δ','')], linewidth=0.9)
    ax.axhline(0, color='black', linewidth=0.6, linestyle='--')
    ax.set_title(f"Log-prinosi – {col.replace('Δlog_','').replace('log_','')}", fontsize=13)
    ax.set_ylabel("Δln(P_t)")

axes[-1].set_xlabel("Datum")
fig.suptitle("Vremenske serije razlika logaritamskih cena (nedeljni prinosi)",
             fontsize=15, fontweight='bold')
plt.tight_layout()
# plt.savefig("log_razlike.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

norm_log = (log_cene - log_cene.mean()) / log_cene.std()
for col in norm_log.columns:
    axes[0].plot(norm_log.index, norm_log[col], label=col.replace('log_',''))
axes[0].set_title("Normalizovane log-cene")
axes[0].set_ylabel("Z-score")
axes[0].legend()

for col in log_razlike.columns:
    axes[1].plot(log_razlike.index, log_razlike[col], alpha=0.7,
                 label=col.replace('Δlog_','').replace('log_',''))
axes[1].set_title("Nedeljni log-prinosi")
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf

def evaluate_white_noise(series, lags=20, alpha=0.05):
    print("=" * 60)
    print("        WEAK WHITE NOISE DIAGNOSTIC REPORT          ")
    print("=" * 60)

    lb_result = acorr_ljungbox(series, lags=[lags], return_df=True)
    lb_pvalue = lb_result['lb_pvalue'].iloc[0]
    bp_pvalue = lb_result['bp_pvalue'].iloc[0] if 'bp_pvalue' in lb_result else None

    print("\n1. AUTOCORRELATION TESTS (Linear Independence)")
    print(f"   • Ljung-Box Test (Lag {lags}): p-value = {lb_pvalue:.4f}")
    if bp_pvalue is not None:
        print(f"   • Box-Pierce Test (Lag {lags}): p-value = {bp_pvalue:.4f}")

    if lb_pvalue > alpha:
        print("   --> RESULT: PASS (No significant autocorrelation detected)")
    else:
        print("   --> RESULT: FAIL (Significant autocorrelation present)")

    adf_result = adfuller(series)
    adf_pvalue = adf_result[1]

    print("\n2. STATIONARITY & CONSTANT MEAN TEST (ADF Test)")
    print(f"   • ADF Statistic: {adf_result[0]:.4f}")
    print(f"   • p-value: {adf_pvalue:.4f}")

    if adf_pvalue < alpha:
        print("   --> RESULT: PASS (Series is stationary around a constant mean)")
    else:
        print("   --> RESULT: FAIL (Unit root detected; mean or trend is shifting)")

    demeaned_series = series - series.mean()
    arch_result = het_arch(demeaned_series, maxlag=lags)
    arch_pvalue = arch_result[1]  # p-value of LM test

    print("\n3. HOMOSCEDASTICITY / CONSTANT VARIANCE (Engle's ARCH Test)")
    print(f"   • ARCH LM p-value: {arch_pvalue:.4f}")

    if arch_pvalue > alpha:
        print("   --> RESULT: PASS (Constant variance / No volatility clustering)")
    else:
        print("   --> RESULT: FAIL (Conditional heteroscedasticity / Volatility clustering present)")

    print("\n" + "-" * 60)
    if lb_pvalue > alpha and adf_pvalue < alpha and arch_pvalue > alpha:
        print("FINAL VERDICT: The series BEHAVES LIKE WEAK WHITE NOISE.")
    elif lb_pvalue > alpha and adf_pvalue < alpha:
        print("FINAL VERDICT: WEAK WHITE NOISE for linear terms, BUT shows non-linear variance dependence (ARCH effects).")
    else:
        print("FINAL VERDICT: The series DOES NOT behave like Weak White Noise.")
    print("-" * 60)

    fig, axes = plt.subplots(3, 1, figsize=(12, 10))
    fig.tight_layout(pad=4.0)

    rolling_window = max(10, len(series) // 20)
    roll_mean = series.rolling(rolling_window).mean()
    roll_std = series.rolling(rolling_window).std()

    axes[0].plot(series.values, label="Data", color="gray", alpha=0.6)
    axes[0].plot(roll_mean.values, label=f"Rolling Mean (win={rolling_window})", color="red", linewidth=1.5)
    axes[0].plot(roll_std.values, label=f"Rolling Std Dev (win={rolling_window})", color="blue", linewidth=1.5)
    axes[0].set_title("Time Series & Rolling Moments")
    axes[0].legend()
    axes[0].grid(True, linestyle=":", alpha=0.6)

    plot_acf(series, lags=lags, ax=axes[1], alpha=alpha)
    axes[1].set_title(f"Autocorrelation Function (ACF) with {int((1-alpha)*100)}% Confidence Bounds")
    axes[1].grid(True, linestyle=":", alpha=0.6)

    axes[2].plot(demeaned_series.values**2, color="purple", alpha=0.7)
    axes[2].set_title("Squared Deviations $(X_t - \\mu)^2$ (Variance Stability Check)")
    axes[2].grid(True, linestyle=":", alpha=0.6)

    plt.show()

In [ ]:
for col in log_razlike.columns:
    print(f"\n=======================================================")
    print(f" TESTIRANJE BELE BUKE ZA: {col}")
    print(f"=======================================================")

    evaluate_white_noise(log_razlike[col], lags=15)

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime


sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# start_date = "2013-01-01"
# end_date   = "2023-01-01"

# tikeri = {
#     "WTI":   "CL=F",
#     "BRENT": "BZ=F",
#     "NVDA":  "NVDA"
# }

# podaci = {}
# for naziv, tiker in tikeri.items():
#     df = yf.download(tiker, start=start_date, end=end_date, interval="1wk",
#                      progress=False, auto_adjust=False)
#     podaci[naziv] = df['Close'].copy()

# cene = pd.concat(podaci, axis=1)
# cene.columns = tikeri.keys()
# cene = cene.dropna()

# print("Prvih 5 redova sirovih cena:")
# print(cene.head())
# print("\nDimenzije:", cene.shape)
# cene = cene[cene > 0].dropna()

# log_cene = np.log(cene)
# log_cene.columns = [f"log_{c}" for c in log_cene.columns]

# log_razlike = log_cene.diff().dropna()
# log_razlike.columns = [f"Δ{col}" for col in log_razlike.columns]

# print("\nPrvih 5 redova log-cena:")
# print(log_cene.head(3))
# print("\nPrvih 5 redova log-razlika (prinosa):")
# print(log_razlike.head(3))

# (Tvoji prethodni grafikoni za cene i razlike su ovde izostavljeni radi uštede prostora,
# ali možeš ih slobodno ostaviti u svom fajlu)

# ==========================================
# IMPLEMENTACIJA AR(1) MODELA I TESTA JEDINIČNOG KORENA
# ==========================================

print("\n" + "="*50)
print("POČETAK AR(1) ANALIZE I TESTA JEDINIČNOG KORENA")
print("="*50 + "\n")

# Funkcija koja izvršava ceo proces na jednom nizu (jednoj koloni iz Pandas DataFrame-a)
def analiziraj_ar1(serija_cena, naziv_instrumenta):
    print(f"--- Analiza za instrument: {naziv_instrumenta} ---")

    # Prebacujemo Pandas Series u NumPy array za brži rad i lakše indeksiranje
    niz_cena = serija_cena.values

    # Korak 1: Priprema podataka (Kreiranje pomaka - Lagging)
    X_t = niz_cena[:-1]
    X_t_plus_1 = niz_cena[1:]

    # Korak 2: Ocenjivanje parametara 'a' i 'b' primenom MNK
    mean_X_t = np.mean(X_t)
    mean_X_t_plus_1 = np.mean(X_t_plus_1)

    brojilac = np.sum((X_t - mean_X_t) * (X_t_plus_1 - mean_X_t_plus_1))
    imenilac = np.sum((X_t - mean_X_t)**2)
    b = brojilac / imenilac
    a = mean_X_t_plus_1 - b * mean_X_t

    print(f"Ocenjeni parametri:")
    print(f"b = {b:.6f}")
    print(f"a = {a:.6f}")

    # Korak 3: Izdvajanje i provera reziduala (e_t)
    e_t = X_t_plus_1 - (a + b * X_t)
    mean_e_t = np.mean(e_t)
    var_e_t = np.var(e_t)

    print(f"Osobine reziduala:")
    print(f"Srednja vrednost e_t: {mean_e_t:.10f}")
    print(f"Srednja varijansa e_t: {var_e_t:.6f}")

    # Vizuelna provera reziduala
    # AKO INSTALIRAŠ statsmodels, PROMENI SLEDEĆI BLOK DA CRTA I ACF
    plt.figure(figsize=(10, 4))
    plt.plot(e_t, color='blue', alpha=0.7)
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f'Reziduali ($e_t$) za {naziv_instrumenta}')
    plt.xlabel('Vreme $t$ (Nedelje)')
    plt.ylabel('Vrednost greške')
    plt.grid(True, alpha=0.3)
    #plt.show()

    # Korak 4: Interpretacija (Test jediničnog korena)
    tolerancija = 0.01

    if abs(b) > 1 + tolerancija:
         zakljucak = "Proces divergira."
    elif abs(b - 1) <= tolerancija:
         zakljucak = "Proces je random wlak."
    elif abs(b) < 1:
         zakljucak = "Proces je kovarijansno stacioniran."
    else:
         zakljucak = "Granični slučaj koji zahteva dodatna ispitivanja."

    print(f"ZAKLJUČAK TESTA: {zakljucak}\n")

# Pokrećemo analizu za svaku od tri log-cene iz tvog DataFrame-a
for kolona in log_cene.columns:
    # Izvlačimo naziv bez 'log_' prefiksa radi lepšeg ispisa
    naziv = kolona.replace("log_", "")
    analiziraj_ar1(log_cene[kolona], naziv)

In [ ]:
"""
=============================================================================
DEO 4 — Random Walk model + Monte Carlo simulacija (WTI nafta)
Samostalni .py fajl (sam preuzima podatke preko yfinance).
=============================================================================
Model:  X_t = X_{t-1} + e_t,  e_t ~ N(mu, sigma^2)
(A) provera da li su reziduali e_t Gaussovi -> ispada da NISU (fat tails)
(B) Monte Carlo simulacija buducnosti        -> lepeza se siri, lose prati stvarnost
=============================================================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import yfinance as yf

# ---------------------------------------------------------------------------
# PREUZIMANJE PODATAKA
# ---------------------------------------------------------------------------
cena = yf.download("CL=F", start="2013-01-01", end="2023-01-01",
                   interval="1wk", progress=False, auto_adjust=False)["Close"]

# 'Close' sa jednim tikerom ume da bude DataFrame sa kolonom 'CL=F' -> svedi na Series
if isinstance(cena, pd.DataFrame):
    cena = cena.iloc[:, 0]

cena = cena.dropna()
cena = cena[cena > 0]

log_WTI = np.log(cena)
log_WTI.name = "log_WTI"

log_ret_WTI = log_WTI.diff().dropna()
log_ret_WTI.name = "dlog_WTI"


# ===========================================================================
# DEO 4A — Da li su reziduali e_t = X_t - X_{t-1} Gaussov beli sum?
#   - histogram vs. teorijska normalna kriva
#   - QQ-plot (ako je normalno, tacke leze na pravoj)
#   - skewness i excess kurtosis (normalna: 0 i 0; fat tails -> kurtosis >> 0)
#   - Jarque-Bera test: p < 0.05 => odbacujemo normalnost
# ===========================================================================
e = log_ret_WTI.dropna()
mu, sigma = e.mean(), e.std()

skew = stats.skew(e)
exc_kurt = stats.kurtosis(e, fisher=True)      # Fisher: normalna -> 0
jb_stat, jb_p = stats.jarque_bera(e)

print("===== DEO 4A: provera Gaussovosti reziduala =====")
print(f"broj odbiraka:      {len(e)}")
print(f"srednja vrednost:   {mu:.5f}")
print(f"std. devijacija:    {sigma:.5f}")
print(f"skewness:           {skew:.3f}   (normalna = 0)")
print(f"excess kurtosis:    {exc_kurt:.3f}   (normalna = 0; >0 znaci fat tails)")
print(f"Jarque-Bera:        stat={jb_stat:.1f},  p-value={jb_p:.2e}")
print("zakljucak:         ", "NIJE normalna (fat tails)" if jb_p < 0.05
      else "ne mozemo odbaciti normalnost")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.hist(e, bins=50, density=True, alpha=0.6, color="steelblue",
         edgecolor="white", label="empirijski")
x = np.linspace(e.min(), e.max(), 300)
ax1.plot(x, stats.norm.pdf(x, mu, sigma), "r-", linewidth=2,
         label=f"N(mu={mu:.3f}, sigma={sigma:.3f})")
ax1.set_title("Histogram log-prinosa vs. normalna raspodela")
ax1.set_xlabel("delta log cena"); ax1.set_ylabel("gustina"); ax1.legend()

stats.probplot(e, dist="norm", plot=ax2)
ax2.set_title("QQ-plot (tacke van prave = fat tails)")

plt.tight_layout()
plt.show()


# ===========================================================================
# DEO 4B — Monte Carlo simulacija random walk modela
#   X_t = X_{t-1} + e_t,  e_t ~ N(mu, sigma^2)
#   Simuliramo M putanja, svaku H nedelja unapred, od poslednje log cene X0.
# ===========================================================================
H = 52        # horizont: 52 nedelje unapred
M = 1000      # broj Monte Carlo putanja
X0 = log_WTI.iloc[-1]

rng = np.random.default_rng(0)                   # fiksan seed radi ponovljivosti
sokovi = rng.normal(mu, sigma, size=(H, M))      # H x M: stubac = putanja
putanje_log = X0 + np.cumsum(sokovi, axis=0)     # random walk = kumulativna suma sokova
putanje_cena = np.exp(putanje_log)               # nazad iz log u cene

sredina = putanje_cena.mean(axis=1)
p05 = np.percentile(putanje_cena, 5, axis=1)
p95 = np.percentile(putanje_cena, 95, axis=1)

nedelje = np.arange(1, H + 1)
cena0 = float(np.exp(X0))

print("\n===== DEO 4B: Monte Carlo random walk =====")
print(f"pocetna cena X0:    {cena0:.2f}")
print(f"horizont:           {H} nedelja,  broj putanja: {M}")
print(f"ned. 52 -> prosek:  {sredina[-1]:.2f}")
print(f"ned. 52 -> 90% int: [{p05[-1]:.2f}, {p95[-1]:.2f}]")
print(f"sirina opsega ned.1:  {p95[0]-p05[0]:.2f}")
print(f"sirina opsega ned.52: {p95[-1]-p05[-1]:.2f}   (siri se ~ sqrt(t))")

plt.figure(figsize=(14, 7))
for j in range(min(100, M)):
    plt.plot(nedelje, putanje_cena[:, j], color="gray", alpha=0.08, linewidth=0.6)
plt.plot(nedelje, sredina, "b-", linewidth=2, label="prosek simulacija")
plt.fill_between(nedelje, p05, p95, color="steelblue", alpha=0.25,
                 label="90% interval (5-95 percentil)")
plt.scatter([0], [cena0], color="black", zorder=5, label="poslednja poznata cena")
plt.title("Monte Carlo simulacija - Random Walk model (WTI, 52 nedelje unapred)")
plt.xlabel("nedelja u buducnost"); plt.ylabel("cena WTI")
plt.legend(); plt.tight_layout()
plt.show()


# ===========================================================================
# (opciono) POREDJENJE SA STVARNOM REALIZACIJOM
# Poslednjih 10 nedeljnih cena WTI od 5. marta 2025.
# ===========================================================================
# stvarne = yf.download("CL=F", start="2025-03-05", interval="1wk",
#                       progress=False, auto_adjust=False)["Close"]
# if isinstance(stvarne, pd.DataFrame):
#     stvarne = stvarne.iloc[:, 0]
# stvarne = stvarne.dropna().iloc[:10]
# plt.figure(figsize=(12, 6))
# plt.plot(range(1, len(stvarne) + 1), stvarne.values, "r-o", label="stvarna cena (2025)")
# plt.plot(nedelje[:10], sredina[:10], "b--", label="prosek simulacija")
# plt.fill_between(nedelje[:10], p05[:10], p95[:10], alpha=0.2)
# plt.legend(); plt.title("Simulacija vs. stvarnost - prvih 10 nedelja")
# plt.tight_layout(); plt.show()

print("\nGOTOVO. Deo 4 kompletan.")